In [1]:
from lcdb.db import LCDB
from lcdb.analysis import LearningCurveExtractor

from pathlib import Path

from tqdm import tqdm

import pandas as pd

In [3]:
# retrieve learning curve objects
lcdb = LCDB()

def compute_payload(row):
    return len(str(row["m:json"]))

for workflow_class in [
    "lcdb.workflow.sklearn.KNNWorkflow",
    "lcdb.workflow.sklearn.LibLinearWorkflow",
    "lcdb.workflow.sklearn.LibSVMWorkflow",
    "lcdb.workflow.sklearn.TreesEnsembleWorkflow",
    "lcdb.workflow.xgboost.XGBoostWorkflow"
]:

    file = Path(f"{workflow_class}.csv")
    if not file.exists():

        print(workflow_class)


        gen = lcdb.query(
            campaigns=["pre-config-100"],
            workflows=[workflow_class],
            test_seeds=[0],
            return_generator=True,
            processors={
                "learning_curve": LearningCurveExtractor(metrics=["error_rate"]),
                "payload": compute_payload
            },
            show_progress=True
        )

        # get all dataframes
        dfs = []
        for chunk_df in tqdm(gen):
            dfs.append(chunk_df)
        df = pd.concat(dfs)
        
        # serialize learning curves
        df["learning_curve"] = df["learning_curve"].apply(lambda lc: lc.to_json() if lc is not None else None)

        df.to_csv(file, index=False)